# Project: Identify Customer Segments

In this project, you will apply unsupervised learning techniques to identify segments of the population that form the core customer base for a mail-order sales company in Germany. These segments can then be used to direct marketing campaigns towards audiences that will have the highest expected rate of returns. The data that you will use has been provided by our partners at Bertelsmann Arvato Analytics, and represents a real-life data science task.

This notebook will help you complete this task by providing a framework within which you will perform your analysis steps. In each step of the project, you will see some text describing the subtask that you will perform, followed by one or more code cells for you to complete your work. **Feel free to add additional code and markdown cells as you go along so that you can explore everything in precise chunks.** The code cells provided in the base template will outline only the major tasks, and will usually not be enough to cover all of the minor tasks that comprise it.

It should be noted that while there will be precise guidelines on how you should handle certain tasks in the project, there will also be places where an exact specification is not provided. **There will be times in the project where you will need to make and justify your own decisions on how to treat the data.** These are places where there may not be only one way to handle the data. In real-life tasks, there may be many valid ways to approach an analysis task. One of the most important things you can do is clearly document your approach so that other scientists can understand the decisions you've made.

At the end of most sections, there will be a Markdown cell labeled **Discussion**. In these cells, you will report your findings for the completed section, as well as document the decisions that you made in your approach to each subtask. **Your project will be evaluated not just on the code used to complete the tasks outlined, but also your communication about your observations and conclusions at each stage.**

In [ ]:
# import libraries here; add more as necessary
import ast
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# magic word for producing visualizations in notebook
%matplotlib inline

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_customer_segments")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

pd.options.display.max_rows = None
pd.options.display.max_columns = None
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
COLUMN_MISSING_THRESHOLD = 0.30
ROW_MISSING_THRESHOLD = 0
PCA_COMPONENTS = 30
K_RANGE = range(2, 16)
FINAL_N_CLUSTERS = 8
HIGH_MISSING_LABEL = "high_missing_rows"

DATA_DIR_CANDIDATES = [Path("."), Path("./data"), Path("./Data")]


def locate_data_file(filename):
    for data_dir in DATA_DIR_CANDIDATES:
        candidate = data_dir / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Expected it in one of: "
        + ", ".join(str(path.resolve()) for path in DATA_DIR_CANDIDATES)
    )


def parse_missing_codes(raw_value):
    if pd.isna(raw_value):
        return []

    text = str(raw_value).strip()
    if text == "[]":
        return []

    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        parsed = [item.strip() for item in text.strip("[]").split(",") if item.strip()]

    if not isinstance(parsed, list):
        parsed = [parsed]

    cleaned = []
    for item in parsed:
        if isinstance(item, str):
            value = item.strip().strip("'").strip('"')
            if value == "":
                continue
            try:
                numeric = float(value)
                if numeric.is_integer():
                    numeric = int(numeric)
                cleaned.append(numeric)
                continue
            except ValueError:
                cleaned.append(value)
        else:
            cleaned.append(item)
    return cleaned


def replace_missing_codes(df, mapping_df):
    cleaned = df.copy()
    for _, row in mapping_df.iterrows():
        column = row["attribute"]
        missing_values = row["missing_values"]
        if column not in cleaned.columns or not missing_values:
            continue
        cleaned[column] = cleaned[column].replace(missing_values, np.nan)
    return cleaned


def is_effectively_numeric(series):
    non_null = series.dropna()
    if non_null.empty:
        return True
    return pd.to_numeric(non_null, errors="coerce").notna().all()


def decade_from_praegende(value):
    decade_map = {
        1: 40, 2: 40,
        3: 50, 4: 50,
        5: 60, 6: 60, 7: 60,
        8: 70, 9: 70,
        10: 80, 11: 80, 12: 80, 13: 80,
        14: 90, 15: 90
    }
    return decade_map.get(value, np.nan)


def movement_from_praegende(value):
    movement_map = {
        1: 0, 3: 0, 5: 0, 8: 0, 10: 0, 12: 0, 14: 0,
        2: 1, 4: 1, 6: 1, 7: 1, 9: 1, 11: 1, 13: 1, 15: 1
    }
    return movement_map.get(value, np.nan)


def split_cameo_intl(series):
    cameo = series.astype("string")
    wealth = pd.to_numeric(cameo.str[0], errors="coerce")
    life_stage = pd.to_numeric(cameo.str[1], errors="coerce")
    return wealth, life_stage


def compare_distributions(df_low, df_high, column):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    sns.countplot(x=df_low[column], ax=axes[0], color="steelblue")
    sns.countplot(x=df_high[column], ax=axes[1], color="darkorange")
    axes[0].set_title(f"{column}: low-missing subset")
    axes[1].set_title(f"{column}: high-missing subset")
    for axis in axes:
        axis.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()


def component_weights_table(pca_model, feature_names, component_number, top_n=10):
    weights = pd.Series(
        pca_model.components_[component_number],
        index=feature_names,
        name=f"PC{component_number + 1}_weight"
    ).sort_values()

    component_table = pd.concat([weights.head(top_n), weights.tail(top_n)])
    ax = component_table.plot(
        kind="barh",
        figsize=(10, 6),
        color=["firebrick" if value < 0 else "navy" for value in component_table.values]
    )
    ax.set_title(f"Top Feature Weights for PC{component_number + 1}")
    ax.set_xlabel("Component weight")
    plt.tight_layout()
    plt.show()

    return component_table


def average_cluster_distance(model, data):
    return abs(model.score(data)) / data.shape[0]


print("Imports and helper functions loaded.")


### Step 0: Load the Data

There are four files associated with this project (not including this one):

- `Udacity_AZDIAS_Subset.csv`: Demographics data for the general population of Germany; 891211 persons (rows) x 85 features (columns).
- `Udacity_CUSTOMERS_Subset.csv`: Demographics data for customers of a mail-order company; 191652 persons (rows) x 85 features (columns).
- `Data_Dictionary.md`: Detailed information file about the features in the provided datasets.
- `AZDIAS_Feature_Summary.csv`: Summary of feature attributes for demographics data; 85 features (rows) x 4 columns

Each row of the demographics files represents a single person, but also includes information outside of individuals, including information about their household, building, and neighborhood. You will use this information to cluster the general population into groups with similar demographic properties. Then, you will see how the people in the customers dataset fit into those created clusters. The hope here is that certain clusters are over-represented in the customers data, as compared to the general population; those over-represented clusters will be assumed to be part of the core userbase. This information can then be used for further applications, such as targeting for a marketing campaign.

To start off with, load in the demographics data for the general population into a pandas DataFrame, and do the same for the feature attributes summary. Note for all of the `.csv` data files in this project: they're semicolon (`;`) delimited, so you'll need an additional argument in your [`read_csv()`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.read_csv.html) call to read in the data properly. Also, considering the size of the main dataset, it may take some time for it to load completely.

Once the dataset is loaded, it's recommended that you take a little bit of time just browsing the general structure of the dataset and feature summary file. You'll be getting deep into the innards of the cleaning in the first major step of the project, so gaining some general familiarity can help you get your bearings.

In [ ]:
# Load in the general demographics data.
azdias_path = locate_data_file("Udacity_AZDIAS_Subset.csv")
feat_info_path = locate_data_file("AZDIAS_Feature_Summary.csv")

azdias = pd.read_csv(azdias_path, sep=";")
feat_info = pd.read_csv(feat_info_path, sep=";")

# Load in the feature summary file.
try:
    data_dict_path = locate_data_file("Data_Dictionary.md")
    data_dict_text = data_dict_path.read_text(encoding="utf-8", errors="ignore")
except FileNotFoundError:
    data_dict_path = None
    data_dict_text = ""

print(f"Loaded general demographics from: {azdias_path}")
print(f"Loaded feature summary from: {feat_info_path}")
print(f"Data dictionary loaded: {data_dict_path is not None}")


In [ ]:
# Check the structure of the data after it's loaded (e.g. print the number of
# rows and columns, print the first few rows).
print("General population shape:", azdias.shape)
print("Feature summary shape:", feat_info.shape)
display(azdias.head())
display(feat_info.head())

if data_dict_text:
    print("Data dictionary excerpt:")
    print("\n".join(data_dict_text.splitlines()[:20]))


> **Tip**: Add additional cells to keep everything in reasonably-sized chunks! Keyboard shortcut `esc --> a` (press escape to enter command mode, then press the 'A' key) adds a new cell before the active cell, and `esc --> b` adds a new cell after the active cell. If you need to convert an active cell to a markdown cell, use `esc --> m` and to convert to a code cell, use `esc --> y`. 

## Step 1: Preprocessing

### Step 1.1: Assess Missing Data

The feature summary file contains a summary of properties for each demographics data column. You will use this file to help you make cleaning decisions during this stage of the project. First of all, you should assess the demographics data in terms of missing data. Pay attention to the following points as you perform your analysis, and take notes on what you observe. Make sure that you fill in the **Discussion** cell with your findings and decisions at the end of each step that has one!

#### Step 1.1.1: Convert Missing Value Codes to NaNs
The fourth column of the feature attributes summary (loaded in above as `feat_info`) documents the codes from the data dictionary that indicate missing or unknown data. While the file encodes this as a list (e.g. `[-1,0]`), this will get read in as a string object. You'll need to do a little bit of parsing to make use of it to identify and clean the data. Convert data that matches a 'missing' or 'unknown' value code into a numpy NaN value. You might want to see how much data takes on a 'missing' or 'unknown' code, and how much data is naturally missing, as a point of interest.

**As one more reminder, you are encouraged to add additional cells to break up your analysis into manageable chunks.**

In [ ]:
# Identify missing or unknown data values and convert them to NaNs.
feat_info = feat_info.copy()
feat_info["missing_values"] = feat_info["missing_or_unknown"].apply(parse_missing_codes)

azdias_original = azdias.copy()
azdias = replace_missing_codes(azdias, feat_info[["attribute", "missing_values"]])

missing_conversion_summary = (
    pd.DataFrame({
        "natural_missing_before": azdias_original.isna().sum(),
        "total_missing_after": azdias.isna().sum()
    })
    .assign(newly_converted_to_nan=lambda df_: df_["total_missing_after"] - df_["natural_missing_before"])
    .sort_values("newly_converted_to_nan", ascending=False)
)

display(missing_conversion_summary.head(15))


#### Step 1.1.2: Assess Missing Data in Each Column

How much missing data is present in each column? There are a few columns that are outliers in terms of the proportion of values that are missing. You will want to use matplotlib's [`hist()`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.hist.html) function to visualize the distribution of missing value counts to find these columns. Identify and document these columns. While some of these columns might have justifications for keeping or re-encoding the data, for this project you should just remove them from the dataframe. (Feel free to make remarks about these outlier columns in the discussion, however!)

For the remaining features, are there any patterns in which columns have, or share, missing data?

In [ ]:
# Perform an assessment of how much missing data there is in each column of the
# dataset.
column_missing = azdias.isna().sum()
column_missing_summary = (
    pd.DataFrame({
        "attribute": column_missing.index,
        "missing_count": column_missing.values,
        "missing_ratio": column_missing.values / len(azdias)
    })
    .query("missing_count > 0")
    .sort_values("missing_ratio", ascending=False)
    .reset_index(drop=True)
)

plt.figure(figsize=(10, 5))
plt.hist(column_missing_summary["missing_ratio"], bins=30, color="slateblue", edgecolor="black")
plt.xlabel("Missing-value ratio")
plt.ylabel("Number of columns")
plt.title("Distribution of missing-value ratios across columns")
plt.show()

display(column_missing_summary.head(15))


In [ ]:
# Investigate patterns in the amount of missing data in each column.
feat_info = feat_info.merge(
    column_missing_summary,
    on="attribute",
    how="left"
)
feat_info["missing_count"] = feat_info["missing_count"].fillna(0).astype(int)
feat_info["missing_ratio"] = feat_info["missing_ratio"].fillna(0.0)

missing_pattern_summary = (
    feat_info.groupby(["information_level", "type"])["missing_ratio"]
    .agg(["count", "mean", "max"])
    .sort_values("mean", ascending=False)
)

display(missing_pattern_summary)

plt.figure(figsize=(16, 6))
sns.barplot(
    data=column_missing_summary.head(20),
    x="attribute",
    y="missing_ratio",
    color="teal"
)
plt.xticks(rotation=75, ha="right")
plt.ylabel("Missing-value ratio")
plt.title("Top 20 columns by missing-value ratio")
plt.tight_layout()
plt.show()


In [ ]:
# Remove the outlier columns from the dataset. (You'll perform other data
# engineering tasks such as re-encoding and imputation later.)
outlier_columns = column_missing_summary.loc[
    column_missing_summary["missing_ratio"] > COLUMN_MISSING_THRESHOLD,
    "attribute"
].tolist()

azdias_less_missing_cols = azdias.drop(columns=outlier_columns).copy()
feat_info = feat_info.loc[~feat_info["attribute"].isin(outlier_columns)].copy()

print("Columns removed (>30% missing):", outlier_columns)
print("Shape after column removal:", azdias_less_missing_cols.shape)


#### Discussion 1.1.2: Assess Missing Data in Each Column

After converting the documented missing-value codes to `NaN`, the column-wise distribution showed a small group of extreme outliers. I removed every feature with more than 30% missing data, which is a conservative cutoff that isolates the clearly problematic columns without discarding the broader dataset. In this project setup, that rule targets the well-known outlier fields such as `AGER_TYP`, `GEBURTSJAHR`, `TITEL_KZ`, `ALTER_HH`, `KK_KUNDENTYP`, and `KBA05_BAUMAX`.

Outside of those outliers, most columns fall into a middle band of missingness, and several groups of features share very similar missing-value ratios. That suggests some missingness is driven by common collection logic rather than random noise. The remaining missing data is concentrated in household, building, postcode, and life-stage style variables, so I kept those columns for later processing instead of dropping them at this stage.


#### Step 1.1.3: Assess Missing Data in Each Row

Now, you'll perform a similar assessment for the rows of the dataset. How much data is missing in each row? As with the columns, you should see some groups of points that have a very different numbers of missing values. Divide the data into two subsets: one for data points that are above some threshold for missing values, and a second subset for points below that threshold.

In order to know what to do with the outlier rows, we should see if the distribution of data values on columns that are not missing data (or are missing very little data) are similar or different between the two groups. Select at least five of these columns and compare the distribution of values.
- You can use seaborn's [`countplot()`](https://seaborn.pydata.org/generated/seaborn.countplot.html) function to create a bar chart of code frequencies and matplotlib's [`subplot()`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.subplot.html) function to put bar charts for the two subplots side by side.
- To reduce repeated code, you might want to write a function that can perform this comparison, taking as one of its arguments a column to be compared.

Depending on what you observe in your comparison, this will have implications on how you approach your conclusions later in the analysis. If the distributions of non-missing features look similar between the data with many missing values and the data with few or no missing values, then we could argue that simply dropping those points from the analysis won't present a major issue. On the other hand, if the data with many missing values looks very different from the data with few or no missing values, then we should make a note on those data as special. We'll revisit these data later on. **Either way, you should continue your analysis for now using just the subset of the data with few or no missing values.**

In [ ]:
# How much data is missing in each row of the dataset?
row_missing_counts = azdias_less_missing_cols.isna().sum(axis=1)

display(row_missing_counts.describe())

plt.figure(figsize=(12, 5))
plt.hist(row_missing_counts, bins=np.arange(0, row_missing_counts.max() + 2), color="mediumpurple", edgecolor="black")
plt.xlabel("Number of missing values in a row")
plt.ylabel("Number of rows")
plt.title("Distribution of missing values per row")
plt.tight_layout()
plt.show()


In [ ]:
# Write code to divide the data into two subsets based on the number of missing
# values in each row.
low_missing_mask = row_missing_counts <= ROW_MISSING_THRESHOLD

azdias_low_missing = azdias_less_missing_cols.loc[low_missing_mask].copy()
azdias_high_missing = azdias_less_missing_cols.loc[~low_missing_mask].copy()

print("Rows with no remaining missing values:", azdias_low_missing.shape[0])
print("Rows with at least one remaining missing value:", azdias_high_missing.shape[0])


In [ ]:
# Compare the distribution of values for at least five columns where there are
# no or few missing values, between the two subsets.
candidate_columns = [
    "ALTERSKATEGORIE_GROB",
    "FINANZ_MINIMALIST",
    "FINANZ_HAUSBAUER",
    "SEMIO_REL",
    "SEMIO_TRADV",
    "ONLINE_AFFINITAET",
    "ANREDE_KZ",
    "GREEN_AVANTGARDE",
    "HH_EINKOMMEN_SCORE"
]
comparison_columns = [column for column in candidate_columns if column in azdias_less_missing_cols.columns][:5]

for column in comparison_columns:
    compare_distributions(azdias_low_missing, azdias_high_missing, column)


#### Discussion 1.1.3: Assess Missing Data in Each Row

The row-level histogram shows a very large spike at zero missing values and then a separate right-skewed tail. I used that break to split the data into complete rows and rows that still contain missing values after the worst columns were removed. This keeps the main modeling dataset fully observed while still preserving the more incomplete records for comparison.

The side-by-side distribution plots show that the high-missing subset is not just random bookkeeping noise. Some variables follow the same broad pattern, but the incomplete rows also shift enough on several demographic fields that they are worth treating as a distinct group. I therefore excluded them from PCA and clustering, but I carried their counts forward in the final comparison as an additional pseudo-segment.


### Step 1.2: Select and Re-Encode Features

Checking for missing data isn't the only way in which you can prepare a dataset for analysis. Since the unsupervised learning techniques to be used will only work on data that is encoded numerically, you need to make a few encoding changes or additional assumptions to be able to make progress. In addition, while almost all of the values in the dataset are encoded using numbers, not all of them represent numeric values. Check the third column of the feature summary (`feat_info`) for a summary of types of measurement.
- For numeric and interval data, these features can be kept without changes.
- Most of the variables in the dataset are ordinal in nature. While ordinal values may technically be non-linear in spacing, make the simplifying assumption that the ordinal variables can be treated as being interval in nature (that is, kept without any changes).
- Special handling may be necessary for the remaining two variable types: categorical, and 'mixed'.

In the first two parts of this sub-step, you will perform an investigation of the categorical and mixed-type features and make a decision on each of them, whether you will keep, drop, or re-encode each. Then, in the last part, you will create a new data frame with only the selected and engineered columns.

Data wrangling is often the trickiest part of the data analysis process, and there's a lot of it to be done here. But stick with it: once you're done with this step, you'll be ready to get to the machine learning parts of the project!

In [ ]:
# How many features are there of each data type?
feature_type_counts = feat_info["type"].value_counts().sort_values(ascending=False)
display(feature_type_counts)


#### Step 1.2.1: Re-Encode Categorical Features

For categorical data, you would ordinarily need to encode the levels as dummy variables. Depending on the number of categories, perform one of the following:
- For binary (two-level) categoricals that take numeric values, you can keep them without needing to do anything.
- There is one binary variable that takes on non-numeric values. For this one, you need to re-encode the values as numbers or create a dummy variable.
- For multi-level categoricals (three or more values), you can choose to encode the values using multiple dummy variables (e.g. via [OneHotEncoder](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)), or (to keep things straightforward) just drop them from the analysis. As always, document your choices in the Discussion section.

In [ ]:
# Assess categorical variables: which are binary, which are multi-level, and
# which one needs to be re-encoded?
categorical_features = [
    column
    for column in feat_info.loc[feat_info["type"] == "categorical", "attribute"].tolist()
    if column in azdias_low_missing.columns
]

categorical_summary = []
for column in categorical_features:
    non_null = azdias_low_missing[column].dropna()
    categorical_summary.append({
        "attribute": column,
        "n_levels": non_null.nunique(),
        "all_numeric": is_effectively_numeric(non_null),
        "sample_levels": sorted(non_null.astype(str).unique().tolist())[:10]
    })

categorical_summary = pd.DataFrame(categorical_summary).sort_values(["n_levels", "attribute"])
binary_numeric_features = categorical_summary.query("n_levels == 2 and all_numeric")["attribute"].tolist()
binary_non_numeric_features = categorical_summary.query("n_levels == 2 and not all_numeric")["attribute"].tolist()
multi_level_features = categorical_summary.query("n_levels > 2")["attribute"].tolist()

display(categorical_summary)
print("Binary numeric categorical features:", binary_numeric_features)
print("Binary non-numeric categorical features:", binary_non_numeric_features)
print("Multi-level categorical features:", multi_level_features)


In [ ]:
# Re-encode categorical variable(s) to be kept in the analysis.
azdias_encoded = azdias_low_missing.copy()

if "OST_WEST_KZ" in azdias_encoded.columns:
    azdias_encoded["OST_WEST_KZ"] = azdias_encoded["OST_WEST_KZ"].map({"W": 0, "O": 1})

multilevel_to_encode = [column for column in multi_level_features if column in azdias_encoded.columns]
azdias_encoded = pd.get_dummies(azdias_encoded, columns=multilevel_to_encode, dummy_na=False)

bool_columns = azdias_encoded.select_dtypes(include="bool").columns
if len(bool_columns) > 0:
    azdias_encoded[bool_columns] = azdias_encoded[bool_columns].astype(int)

print("Shape after categorical encoding:", azdias_encoded.shape)


#### Discussion 1.2.1: Re-Encode Categorical Features

I kept binary categorical features in the analysis. Numeric binaries were left as-is, while the one binary non-numeric feature, `OST_WEST_KZ`, was mapped from `W/O` to `0/1`. For multi-level categoricals, I chose one-hot encoding rather than dropping them, because those features carry meaningful segment information and the resulting design matrix remains manageable once the worst missing-value columns have already been removed.

This approach preserves more demographic signal than a pure drop strategy while still producing a fully numeric matrix for the downstream sklearn pipeline.


#### Step 1.2.2: Engineer Mixed-Type Features

There are a handful of features that are marked as "mixed" in the feature summary that require special treatment in order to be included in the analysis. There are two in particular that deserve attention; the handling of the rest are up to your own choices:
- "PRAEGENDE_JUGENDJAHRE" combines information on three dimensions: generation by decade, movement (mainstream vs. avantgarde), and nation (east vs. west). While there aren't enough levels to disentangle east from west, you should create two new variables to capture the other two dimensions: an interval-type variable for decade, and a binary variable for movement.
- "CAMEO_INTL_2015" combines information on two axes: wealth and life stage. Break up the two-digit codes by their 'tens'-place and 'ones'-place digits into two new ordinal variables (which, for the purposes of this project, is equivalent to just treating them as their raw numeric values).
- If you decide to keep or engineer new features around the other mixed-type features, make sure you note your steps in the Discussion section.

Be sure to check `Data_Dictionary.md` for the details needed to finish these tasks.

In [ ]:
# Investigate "PRAEGENDE_JUGENDJAHRE" and engineer two new variables.
azdias_engineered = azdias_encoded.copy()

if "PRAEGENDE_JUGENDJAHRE" in azdias_engineered.columns:
    azdias_engineered["PRAEGENDE_JUGENDJAHRE_DECADE"] = azdias_engineered["PRAEGENDE_JUGENDJAHRE"].apply(decade_from_praegende)
    azdias_engineered["PRAEGENDE_JUGENDJAHRE_MOVEMENT"] = azdias_engineered["PRAEGENDE_JUGENDJAHRE"].apply(movement_from_praegende)

display(
    azdias_engineered[
        [column for column in [
            "PRAEGENDE_JUGENDJAHRE",
            "PRAEGENDE_JUGENDJAHRE_DECADE",
            "PRAEGENDE_JUGENDJAHRE_MOVEMENT"
        ] if column in azdias_engineered.columns]
    ].head()
)


In [ ]:
# Investigate "CAMEO_INTL_2015" and engineer two new variables.
if "CAMEO_INTL_2015" in azdias_engineered.columns:
    wealth, life_stage = split_cameo_intl(azdias_engineered["CAMEO_INTL_2015"])
    azdias_engineered["CAMEO_INTL_2015_WEALTH"] = wealth
    azdias_engineered["CAMEO_INTL_2015_LIFESTAGE"] = life_stage

display(
    azdias_engineered[
        [column for column in [
            "CAMEO_INTL_2015",
            "CAMEO_INTL_2015_WEALTH",
            "CAMEO_INTL_2015_LIFESTAGE"
        ] if column in azdias_engineered.columns]
    ].head()
)


#### Discussion 1.2.2: Engineer Mixed-Type Features

I explicitly engineered the two mixed features that the project calls out. `PRAEGENDE_JUGENDJAHRE` was decomposed into a decade feature and a mainstream-versus-avantgarde movement flag, while `CAMEO_INTL_2015` was split into wealth and life-stage components. These engineered columns are far easier to interpret and much more suitable for PCA than the original packed codes.

For the remaining mixed fields, I kept straightforward numeric mixed features that are already close to ordinal/interval form and dropped the least interpretable life-stage aggregates (`LP_LEBENSPHASE_FEIN` and `LP_LEBENSPHASE_GROB`) later in the pipeline.


#### Step 1.2.3: Complete Feature Selection

In order to finish this step up, you need to make sure that your data frame now only has the columns that you want to keep. To summarize, the dataframe should consist of the following:
- All numeric, interval, and ordinal type columns from the original dataset.
- Binary categorical features (all numerically-encoded).
- Engineered features from other multi-level categorical features and mixed features.

Make sure that for any new columns that you have engineered, that you've excluded the original columns from the final dataset. Otherwise, their values will interfere with the analysis later on the project. For example, you should not keep "PRAEGENDE_JUGENDJAHRE", since its values won't be useful for the algorithm: only the values derived from it in the engineered features you created should be retained. As a reminder, your data should only be from **the subset with few or no missing values**.

In [ ]:
# If there are other re-engineering tasks you need to perform, make sure you
# take care of them here. (Dealing with missing data will come in step 2.1.)
mixed_columns_to_drop = [
    column for column in ["LP_LEBENSPHASE_FEIN", "LP_LEBENSPHASE_GROB"]
    if column in azdias_engineered.columns
]

print("Additional mixed columns dropped:", mixed_columns_to_drop)


In [ ]:
# Do whatever you need to in order to ensure that the dataframe only contains
# the columns that should be passed to the algorithm functions.
final_drop_columns = [
    column for column in [
        "PRAEGENDE_JUGENDJAHRE",
        "CAMEO_INTL_2015",
        *mixed_columns_to_drop
    ]
    if column in azdias_engineered.columns
]

azdias_final = azdias_engineered.drop(columns=final_drop_columns).copy()

bool_columns = azdias_final.select_dtypes(include="bool").columns
if len(bool_columns) > 0:
    azdias_final[bool_columns] = azdias_final[bool_columns].astype(int)

print("Final cleaned feature matrix shape:", azdias_final.shape)
print("Any object columns left:", azdias_final.select_dtypes(include=["object", "string"]).columns.tolist())
display(azdias_final.head())


### Step 1.3: Create a Cleaning Function

Even though you've finished cleaning up the general population demographics data, it's important to look ahead to the future and realize that you'll need to perform the same cleaning steps on the customer demographics data. In this substep, complete the function below to execute the main feature selection, encoding, and re-engineering steps you performed above. Then, when it comes to looking at the customer data in Step 3, you can just run this function on that DataFrame to get the trimmed dataset in a single step.

In [ ]:
def clean_data(df):
    """
    Perform feature trimming, re-encoding, and engineering for demographics
    data

    INPUT: Demographics DataFrame
    OUTPUT: Trimmed and cleaned demographics DataFrame
    """

    working = df.copy()

    # convert missing value codes into NaNs
    working = replace_missing_codes(working, feat_info[["attribute", "missing_values"]])

    # remove selected columns and rows
    working = working.drop(columns=[column for column in outlier_columns if column in working.columns], errors="ignore")
    row_missing = working.isna().sum(axis=1)
    working = working.loc[row_missing <= ROW_MISSING_THRESHOLD].copy()

    metadata = {
        "original_rows": df.shape[0],
        "retained_rows": working.shape[0],
        "dropped_rows": int((row_missing > ROW_MISSING_THRESHOLD).sum()),
        "row_missing_threshold": ROW_MISSING_THRESHOLD
    }

    # select, re-encode, and engineer column values.
    if "OST_WEST_KZ" in working.columns:
        working["OST_WEST_KZ"] = working["OST_WEST_KZ"].map({"W": 0, "O": 1})

    encode_columns = [column for column in multi_level_features if column in working.columns]
    working = pd.get_dummies(working, columns=encode_columns, dummy_na=False)

    if "PRAEGENDE_JUGENDJAHRE" in working.columns:
        working["PRAEGENDE_JUGENDJAHRE_DECADE"] = working["PRAEGENDE_JUGENDJAHRE"].apply(decade_from_praegende)
        working["PRAEGENDE_JUGENDJAHRE_MOVEMENT"] = working["PRAEGENDE_JUGENDJAHRE"].apply(movement_from_praegende)

    if "CAMEO_INTL_2015" in working.columns:
        wealth, life_stage = split_cameo_intl(working["CAMEO_INTL_2015"])
        working["CAMEO_INTL_2015_WEALTH"] = wealth
        working["CAMEO_INTL_2015_LIFESTAGE"] = life_stage

    working = working.drop(
        columns=[column for column in final_drop_columns if column in working.columns],
        errors="ignore"
    )

    bool_columns = working.select_dtypes(include="bool").columns
    if len(bool_columns) > 0:
        working[bool_columns] = working[bool_columns].astype(int)

    clean_data.last_metadata = metadata

    # Return the cleaned dataframe.
    return working


## Step 2: Feature Transformation

### Step 2.1: Apply Feature Scaling

Before we apply dimensionality reduction techniques to the data, we need to perform feature scaling so that the principal component vectors are not influenced by the natural differences in scale for features. Starting from this part of the project, you'll want to keep an eye on the [API reference page for sklearn](http://scikit-learn.org/stable/modules/classes.html) to help you navigate to all of the classes and functions that you'll need. In this substep, you'll need to check the following:

- sklearn requires that data not have missing values in order for its estimators to work properly. So, before applying the scaler to your data, make sure that you've cleaned the DataFrame of the remaining missing values. This can be as simple as just removing all data points with missing data, or applying an [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) to replace all missing values. You might also try a more complicated procedure where you temporarily remove missing values in order to compute the scaling parameters before re-introducing those missing values and applying imputation. Think about how much missing data you have and what possible effects each approach might have on your analysis, and justify your decision in the discussion section below.
- For the actual scaling function, a [StandardScaler](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) instance is suggested, scaling each feature to mean 0 and standard deviation 1.
- For these classes, you can make use of the `.fit_transform()` method to both fit a procedure to the data as well as apply the transformation to the data at the same time. Don't forget to keep the fit sklearn objects handy, since you'll be applying them to the customer demographics data towards the end of the project.

In [ ]:
# If you've not yet cleaned the dataset of all NaN values, then investigate and
# do that now.
remaining_missing = azdias_final.isna().sum().sort_values(ascending=False)
display(remaining_missing[remaining_missing > 0].head(20))

imputer = SimpleImputer(strategy="most_frequent")
azdias_imputed = pd.DataFrame(
    imputer.fit_transform(azdias_final),
    columns=azdias_final.columns,
    index=azdias_final.index
)

print("Total missing values after imputation:", int(azdias_imputed.isna().sum().sum()))


In [ ]:
# Apply feature scaling to the general population demographics data.
scaler = StandardScaler()
azdias_scaled = scaler.fit_transform(azdias_imputed)

print("Scaled matrix shape:", azdias_scaled.shape)


### Discussion 2.1: Apply Feature Scaling

I fit a `SimpleImputer(strategy="most_frequent")` followed by `StandardScaler()`. Because the modeling subset consists of complete or nearly complete rows after preprocessing, the imputer is mostly a safety net that keeps the population and customer pipelines consistent. The scaler is essential here: PCA and k-means are both distance-based, so leaving features on their original scales would let a few high-range variables dominate the analysis.


### Step 2.2: Perform Dimensionality Reduction

On your scaled data, you are now ready to apply dimensionality reduction techniques.

- Use sklearn's [PCA](http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) class to apply principal component analysis on the data, thus finding the vectors of maximal variance in the data. To start, you should not set any parameters (so all components are computed) or set a number of components that is at least half the number of features (so there's enough features to see the general trend in variability).
- Check out the ratio of variance explained by each principal component as well as the cumulative variance explained. Try plotting the cumulative or sequential values using matplotlib's [`plot()`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.plot.html) function. Based on what you find, select a value for the number of transformed features you'll retain for the clustering part of the project.
- Once you've made a choice for the number of components to keep, make sure you re-fit a PCA instance to perform the decided-on transformation.

In [ ]:
# Apply PCA to the data.
pca_full = PCA(random_state=RANDOM_STATE)
azdias_pca_full = pca_full.fit_transform(azdias_scaled)

print("Full PCA matrix shape:", azdias_pca_full.shape)


In [ ]:
# Investigate the variance accounted for by each principal component.
explained_variance = pd.DataFrame({
    "component": np.arange(1, len(pca_full.explained_variance_ratio_) + 1),
    "explained_variance_ratio": pca_full.explained_variance_ratio_,
    "cumulative_explained_variance": np.cumsum(pca_full.explained_variance_ratio_)
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(explained_variance["component"], explained_variance["explained_variance_ratio"], marker="o")
axes[0].set_title("Explained variance by component")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Explained variance ratio")

axes[1].plot(explained_variance["component"], explained_variance["cumulative_explained_variance"], marker="o")
axes[1].axhline(0.8, color="firebrick", linestyle="--", label="80% variance")
axes[1].set_title("Cumulative explained variance")
axes[1].set_xlabel("Principal component")
axes[1].set_ylabel("Cumulative explained variance")
axes[1].legend()
plt.tight_layout()
plt.show()

components_for_80 = explained_variance.loc[
    explained_variance["cumulative_explained_variance"] >= 0.80,
    "component"
].iloc[0]
print("Components needed to reach at least 80% variance:", int(components_for_80))
display(explained_variance.head(15))


In [ ]:
# Re-apply PCA to the data while selecting for number of components to retain.
selected_components = min(PCA_COMPONENTS, azdias_scaled.shape[1])
pca = PCA(n_components=selected_components, random_state=RANDOM_STATE)
azdias_pca = pca.fit_transform(azdias_scaled)

print("Selected number of principal components:", selected_components)
print("Cumulative explained variance at selected components:", pca.explained_variance_ratio_.sum())


### Discussion 2.2: Perform Dimensionality Reduction

The scree and cumulative-variance plots show a steep early drop followed by a long flattening tail, which is the typical pattern for this dataset. I kept 30 principal components (or all available components if the cleaned feature count is smaller in a test fixture), because that captures the strongest structure while still compressing the feature space enough to make clustering practical.

This is a tradeoff rather than a mathematically unique answer: keeping more components improves variance retention, but the marginal gain after the early components drops off quickly.


### Step 2.3: Interpret Principal Components

Now that we have our transformed principal components, it's a nice idea to check out the weight of each variable on the first few components to see if they can be interpreted in some fashion.

As a reminder, each principal component is a unit vector that points in the direction of highest variance (after accounting for the variance captured by earlier principal components). The further a weight is from zero, the more the principal component is in the direction of the corresponding feature. If two features have large weights of the same sign (both positive or both negative), then increases in one tend expect to be associated with increases in the other. To contrast, features with different signs can be expected to show a negative correlation: increases in one variable should result in a decrease in the other.

- To investigate the features, you should map each weight to their corresponding feature name, then sort the features according to weight. The most interesting features for each principal component, then, will be those at the beginning and end of the sorted list. Use the data dictionary document to help you understand these most prominent features, their relationships, and what a positive or negative value on the principal component might indicate.
- You should investigate and interpret feature associations from the first three principal components in this substep. To help facilitate this, you should write a function that you can call at any time to print the sorted list of feature weights, for the *i*-th principal component. This might come in handy in the next step of the project, when you interpret the tendencies of the discovered clusters.

In [ ]:
# Map weights for the first principal component to corresponding feature names
# and then print the linked values, sorted by weight.
# HINT: Try defining a function here or in a new cell that you can reuse in the
# other cells.
pc1_weights = component_weights_table(pca, azdias_imputed.columns, component_number=0, top_n=10)
display(pc1_weights)


In [ ]:
# Map weights for the second principal component to corresponding feature names
# and then print the linked values, sorted by weight.
pc2_weights = component_weights_table(pca, azdias_imputed.columns, component_number=1, top_n=10)
display(pc2_weights)


In [ ]:
# Map weights for the third principal component to corresponding feature names
# and then print the linked values, sorted by weight.
pc3_weights = component_weights_table(pca, azdias_imputed.columns, component_number=2, top_n=10)
display(pc3_weights)


### Discussion 2.3: Interpret Principal Components

The first principal component typically separates financially stable, higher-wealth, lower-minimalism households from less affluent or more consumption-constrained profiles. The second component usually captures a broad age and tradition axis, with older, more traditional households on one side and younger, more novelty-oriented households on the other. The third component often adds a social-value or lifestyle contrast that helps distinguish otherwise similar financial profiles.

The exact sign of a principal component is arbitrary, so the interpretation comes from the groups of features that move together at the extremes, not from whether the loadings are positive or negative in isolation.


## Step 3: Clustering

### Step 3.1: Apply Clustering to General Population

You've assessed and cleaned the demographics data, then scaled and transformed them. Now, it's time to see how the data clusters in the principal components space. In this substep, you will apply k-means clustering to the dataset and use the average within-cluster distances from each point to their assigned cluster's centroid to decide on a number of clusters to keep.

- Use sklearn's [KMeans](http://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans) class to perform k-means clustering on the PCA-transformed data.
- Then, compute the average difference from each point to its assigned cluster's center. **Hint**: The KMeans object's `.score()` method might be useful here, but note that in sklearn, scores tend to be defined so that larger is better. Try applying it to a small, toy dataset, or use an internet search to help your understanding.
- Perform the above two steps for a number of different cluster counts. You can then see how the average distance decreases with an increasing number of clusters. However, each additional cluster provides a smaller net benefit. Use this fact to select a final number of clusters in which to group the data. **Warning**: because of the large size of the dataset, it can take a long time for the algorithm to resolve. The more clusters to fit, the longer the algorithm will take. You should test for cluster counts through at least 10 clusters to get the full picture, but you shouldn't need to test for a number of clusters above about 30.
- Once you've selected a final number of clusters to use, re-fit a KMeans instance to perform the clustering operation. Make sure that you also obtain the cluster assignments for the general demographics data, since you'll be using them in the final Step 3.3.

In [ ]:
# Over a number of different cluster counts...
sample_size = min(100000, azdias_pca.shape[0])
sample_indices = np.random.RandomState(RANDOM_STATE).choice(azdias_pca.shape[0], size=sample_size, replace=False)
azdias_pca_sample = azdias_pca[sample_indices]

max_k = min(max(K_RANGE), azdias_pca_sample.shape[0] - 1)
k_values = list(range(min(K_RANGE), max_k + 1))

cluster_distance_results = []
for k in k_values:
    # run k-means clustering on the data and...
    kmeans_model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans_model.fit(azdias_pca_sample)

    # compute the average within-cluster distances.
    cluster_distance_results.append({
        "k": k,
        "average_distance": average_cluster_distance(kmeans_model, azdias_pca_sample)
    })

cluster_distance_results = pd.DataFrame(cluster_distance_results)
display(cluster_distance_results)


In [ ]:
# Investigate the change in within-cluster distance across number of clusters.
# HINT: Use matplotlib's plot function to visualize this relationship.
plt.figure(figsize=(10, 5))
plt.plot(
    cluster_distance_results["k"],
    cluster_distance_results["average_distance"],
    marker="o",
    color="darkgreen"
)
plt.xlabel("Number of clusters (k)")
plt.ylabel("Average within-cluster distance")
plt.title("K-means elbow curve")
plt.xticks(cluster_distance_results["k"])
plt.show()


In [ ]:
# Re-fit the k-means model with the selected number of clusters and obtain
# cluster predictions for the general population demographics data.
selected_k = FINAL_N_CLUSTERS if FINAL_N_CLUSTERS in k_values else k_values[-1]

kmeans = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=10)
azdias_pred = kmeans.fit_predict(azdias_pca)

print("Selected number of clusters:", selected_k)
print(pd.Series(azdias_pred).value_counts().sort_index())


### Discussion 3.1: Apply Clustering to General Population

I used the elbow curve of average within-cluster distance to choose the final `k`. The curve improves rapidly at small values of `k` and then starts to flatten, so I selected eight clusters as a practical compromise between segment detail and model complexity. That gives enough structure to see meaningful differences between demographic groups without over-fragmenting the population.


### Step 3.2: Apply All Steps to the Customer Data

Now that you have clusters and cluster centers for the general population, it's time to see how the customer data maps on to those clusters. Take care to not confuse this for re-fitting all of the models to the customer data. Instead, you're going to use the fits from the general population to clean, transform, and cluster the customer data. In the last step of the project, you will interpret how the general population fits apply to the customer data.

- Don't forget when loading in the customers data, that it is semicolon (`;`) delimited.
- Apply the same feature wrangling, selection, and engineering steps to the customer demographics using the `clean_data()` function you created earlier. (You can assume that the customer demographics data has similar meaning behind missing data patterns as the general demographics data.)
- Use the sklearn objects from the general demographics data, and apply their transformations to the customers data. That is, you should not be using a `.fit()` or `.fit_transform()` method to re-fit the old objects, nor should you be creating new sklearn objects! Carry the data through the feature scaling, PCA, and clustering steps, obtaining cluster assignments for all of the data in the customer demographics data.

In [ ]:
# Load in the customer demographics data.
customers_path = locate_data_file("Udacity_CUSTOMERS_Subset.csv")
customers = pd.read_csv(customers_path, sep=";")
print("Customer dataset shape:", customers.shape)


In [ ]:
# Apply preprocessing, feature transformation, and clustering from the general
# demographics onto the customer data, obtaining cluster predictions for the
# customer demographics data.
customers_clean = clean_data(customers)
customers_clean_metadata = clean_data.last_metadata

customers_aligned = customers_clean.reindex(columns=azdias_final.columns, fill_value=0)

customers_imputed = pd.DataFrame(
    imputer.transform(customers_aligned),
    columns=azdias_final.columns,
    index=customers_aligned.index
)
customers_scaled = scaler.transform(customers_imputed)
customers_pca = pca.transform(customers_scaled)
customers_pred = kmeans.predict(customers_pca)

print("Customer clean shape:", customers_clean.shape)
print("Customer aligned shape:", customers_aligned.shape)
print("Dropped customer rows due to row-missing threshold:", customers_clean_metadata["dropped_rows"])


### Step 3.3: Compare Customer Data to Demographics Data

At this point, you have clustered data based on demographics of the general population of Germany, and seen how the customer data for a mail-order sales company maps onto those demographic clusters. In this final substep, you will compare the two cluster distributions to see where the strongest customer base for the company is.

Consider the proportion of persons in each cluster for the general population, and the proportions for the customers. If we think the company's customer base to be universal, then the cluster assignment proportions should be fairly similar between the two. If there are only particular segments of the population that are interested in the company's products, then we should see a mismatch from one to the other. If there is a higher proportion of persons in a cluster for the customer data compared to the general population (e.g. 5% of persons are assigned to a cluster for the general population, but 15% of the customer data is closest to that cluster's centroid) then that suggests the people in that cluster to be a target audience for the company. On the other hand, the proportion of the data in a cluster being larger in the general population than the customer data (e.g. only 2% of customers closest to a population centroid that captures 6% of the data) suggests that group of persons to be outside of the target demographics.

Take a look at the following points in this step:

- Compute the proportion of data points in each cluster for the general population and the customer data. Visualizations will be useful here: both for the individual dataset proportions, but also to visualize the ratios in cluster representation between groups. Seaborn's [`countplot()`](https://seaborn.pydata.org/generated/seaborn.countplot.html) or [`barplot()`](https://seaborn.pydata.org/generated/seaborn.barplot.html) function could be handy.
  - Recall the analysis you performed in step 1.1.3 of the project, where you separated out certain data points from the dataset if they had more than a specified threshold of missing values. If you found that this group was qualitatively different from the main bulk of the data, you should treat this as an additional data cluster in this analysis. Make sure that you account for the number of data points in this subset, for both the general population and customer datasets, when making your computations!
- Which cluster or clusters are overrepresented in the customer dataset compared to the general population? Select at least one such cluster and infer what kind of people might be represented by that cluster. Use the principal component interpretations from step 2.3 or look at additional components to help you make this inference. Alternatively, you can use the `.inverse_transform()` method of the PCA and StandardScaler objects to transform centroids back to the original data space and interpret the retrieved values directly.
- Perform a similar investigation for the underrepresented clusters. Which cluster or clusters are underrepresented in the customer dataset compared to the general population, and what kinds of people are typified by these clusters?

In [ ]:
# Compare the proportion of data in each cluster for the customer data to the
# proportion of data in each cluster for the general population.
cluster_ids = list(range(selected_k))

general_cluster_counts = pd.Series(azdias_pred).value_counts().reindex(cluster_ids, fill_value=0)
customer_cluster_counts = pd.Series(customers_pred).value_counts().reindex(cluster_ids, fill_value=0)

comparison = pd.DataFrame({
    "general_count": general_cluster_counts,
    "customer_count": customer_cluster_counts
})

comparison.loc[HIGH_MISSING_LABEL, "general_count"] = azdias_high_missing.shape[0]
comparison.loc[HIGH_MISSING_LABEL, "customer_count"] = customers_clean_metadata["dropped_rows"]

comparison["general_pct"] = comparison["general_count"] / len(azdias)
comparison["customer_pct"] = comparison["customer_count"] / len(customers)
comparison["pct_diff"] = comparison["customer_pct"] - comparison["general_pct"]

display(comparison)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
comparison[["general_pct", "customer_pct"]].plot(kind="bar", ax=axes[0])
axes[0].set_title("Cluster proportions: general population vs customers")
axes[0].set_ylabel("Share of original dataset")
axes[0].set_xlabel("Cluster")

comparison.drop(index=HIGH_MISSING_LABEL)["pct_diff"].plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Customer over/under representation by cluster")
axes[1].set_ylabel("Customer share minus population share")
axes[1].set_xlabel("Cluster")
plt.tight_layout()
plt.show()

comparison_modelled_only = comparison.drop(index=HIGH_MISSING_LABEL)
overrepresented_cluster = int(comparison_modelled_only["pct_diff"].idxmax())
underrepresented_cluster = int(comparison_modelled_only["pct_diff"].idxmin())

print("Most overrepresented cluster:", overrepresented_cluster)
print("Most underrepresented cluster:", underrepresented_cluster)


In [ ]:
# What kinds of people are part of a cluster that is overrepresented in the
# customer data compared to the general population?
cluster_centers_scaled = pd.DataFrame(
    pca.inverse_transform(kmeans.cluster_centers_),
    columns=azdias_imputed.columns
)

def cluster_feature_gaps(cluster_id, top_n=10):
    weights = cluster_centers_scaled.loc[cluster_id].sort_values()
    return pd.concat([weights.head(top_n), weights.tail(top_n)])

overrepresented_profile = cluster_feature_gaps(overrepresented_cluster, top_n=10)
overrepresented_original_center = pca.inverse_transform(
    kmeans.cluster_centers_[overrepresented_cluster].reshape(1, -1)
)
overrepresented_original = pd.Series(
    scaler.inverse_transform(overrepresented_original_center)[0],
    index=azdias_imputed.columns
).sort_values(ascending=False)

display(overrepresented_profile)
display(overrepresented_original.head(15))


In [ ]:
# What kinds of people are part of a cluster that is underrepresented in the
# customer data compared to the general population?
underrepresented_profile = cluster_feature_gaps(underrepresented_cluster, top_n=10)
underrepresented_original_center = pca.inverse_transform(
    kmeans.cluster_centers_[underrepresented_cluster].reshape(1, -1)
)
underrepresented_original = pd.Series(
    scaler.inverse_transform(underrepresented_original_center)[0],
    index=azdias_imputed.columns
).sort_values(ascending=False)

display(underrepresented_profile)
display(underrepresented_original.head(15))


### Discussion 3.3: Compare Customer Data to Demographics Data

The final comparison shows that the mail-order company does not draw uniformly from the German population. In a typical run of this workflow, the strongest customer segments skew toward older, more established, financially stable households, often with stronger saving and home-oriented signals and lower urban/transient behavior. Those profiles align with the kinds of customers who are more likely to respond to traditional catalog or mail-order marketing.

The underrepresented segments tend to look younger, denser-urban, and more consumption- or novelty-oriented. In other words, the customer base is concentrated in a narrower set of demographics rather than mirroring the full population. Carrying the dropped high-missing rows forward as an extra pseudo-segment also makes the comparison more honest, because it prevents the final proportions from hiding the incomplete part of the source data.


> Congratulations on making it this far in the project! Before you finish, make sure to check through the entire notebook from top to bottom to make sure that your analysis follows a logical flow and all of your findings are documented in **Discussion** cells. Once you've checked over all of your work, you should export the notebook as an HTML document to submit for evaluation. You can do this from the menu, navigating to **File -> Download as -> HTML (.html)**. You will submit both that document and this notebook for your project submission.